# Posterior credible intervals for the number of cells with positive/negative annual snow trends

This notebook estimates uncertainty in the Table 1 cell counts. Table 1 classifies a cell by the sign of its posterior median annual trend, which produces a point count. Here, for every posterior draw, we count how many of the 1,618 cells have positive or negative annual trend. Quantiles of those draw-level counts provide posterior credible intervals.

For draw $m$ and cell $s$, the annualized trend is

$$R_{annual}^{(m)}(s) = \frac{\sum_{w=1}^{52} P_{final}^{(m)}(s,w)-\sum_{w=1}^{52}P_{initial}^{(m)}(s,w)}{51}.$$

The full trend prediction archives were generated by `py_models/trend_pred.ipynb`. The original Table 1 point-count code is in `Application/trend_pred/trend_agg_2.R` (`make_table_row`).

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path(r'path/to/snow/data-and-results')
OUT_DIR = Path(r'path/to/Snow-Trend\Application\trend_pred')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    'IND': 'trend_ind_full.npz',
    'BYM': 'trend_bym_weekly_full.npz',
    'BYM+': 'trend_weekly_bym+cov+lon.npz',
}
N_YEARS_MINUS_ONE = 51
CI_PROBS = (0.025, 0.975)
print('Data directory:', BASE_DIR)
print('All inputs present:', all((BASE_DIR / f).exists() for f in MODEL_FILES.values()))

## Compute draw-level annual trends and cell counts

The archives are compressed and each probability array is about 4.2 GB uncompressed. The loader processes `weekly_ini` and `weekly_final` sequentially to keep peak memory modest.

In [ ]:
def load_annual_trend_draws(npz_path):
    with np.load(npz_path, allow_pickle=False) as archive:
        ini = archive['weekly_ini']
        if ini.ndim != 4 or ini.shape[2:] != (52, 2):
            raise ValueError(f'Unexpected weekly_ini shape: {ini.shape}')
        ini_sum = ini[:, :, :, 1].sum(axis=2)
        shape = ini.shape
        del ini
        gc.collect()

        final = archive['weekly_final']
        if final.shape != shape:
            raise ValueError(f'weekly_final shape {final.shape} != weekly_ini shape {shape}')
        final_sum = final[:, :, :, 1].sum(axis=2)
        del final
        gc.collect()

    annual_trend = (final_sum - ini_sum) / N_YEARS_MINUS_ONE
    return annual_trend

def summarize_model(model, npz_path):
    print(f'Loading {model}: {npz_path.name}')
    trend_draws = load_annual_trend_draws(npz_path)
    n_draws, n_cells = trend_draws.shape

    positive_counts = (trend_draws > 0).sum(axis=1)
    negative_counts = (trend_draws < 0).sum(axis=1)
    zero_counts = (trend_draws == 0).sum(axis=1)

    # This exactly matches the sign rule used for the original Table 1 point counts.
    cell_posterior_median = np.quantile(trend_draws, 0.5, axis=0)
    table1_positive = int((cell_posterior_median > 0).sum())
    table1_negative = int((cell_posterior_median < 0).sum())

    result = {
        'model': model,
        'n_draws': n_draws,
        'n_cells': n_cells,
        'positive_counts': positive_counts,
        'negative_counts': negative_counts,
        'zero_counts': zero_counts,
        'table1_positive': table1_positive,
        'table1_negative': table1_negative,
    }
    del trend_draws, cell_posterior_median
    gc.collect()
    return result

results = {}
for model, filename in MODEL_FILES.items():
    results[model] = summarize_model(model, BASE_DIR / filename)
print('Finished all models.')

## Posterior summaries

`Table 1 positive/negative` reproduces the manuscript's point classification. `Posterior count median` and `95% CrI` summarize the draw-level number of cells with positive/negative trend used for interval estimation.

In [ ]:
def count_summary(x, n_cells):
    q025, median, q975 = np.quantile(x, [0.025, 0.5, 0.975], method='nearest')
    return {
        'posterior_mean_count': x.mean(),
        'posterior_median_count': median,
        'count_CrI_2.5%': q025,
        'count_CrI_97.5%': q975,
        'posterior_mean_percent': 100 * x.mean() / n_cells,
        'percent_CrI_2.5%': 100 * q025 / n_cells,
        'percent_CrI_97.5%': 100 * q975 / n_cells,
    }

rows = []
for model, r in results.items():
    for direction, key, point_key in [
        ('Positive', 'positive_counts', 'table1_positive'),
        ('Negative', 'negative_counts', 'table1_negative'),
    ]:
        row = {
            'Model': model,
            'Direction': direction,
            'Table_1_point_count': r[point_key],
            'Table_1_point_percent': 100 * r[point_key] / r['n_cells'],
        }
        row.update(count_summary(r[key], r['n_cells']))
        rows.append(row)

summary = pd.DataFrame(rows)
display(summary.round(3))
summary.to_csv(OUT_DIR / 'table1_count_credible_intervals.csv', index=False)